In [1]:
import soundfile as sf
import numpy as np
import pandas as pd
import glob
import os
sampled_root = '../data/sampled'


In [2]:
df = pd.read_csv('dataHasilConvert.csv')
df

,flac_path,wav_path,format_flac,format_wav,subtype_flac,subtype_wav,sr,duration_s,size_flac_kb,size_wav_kb
0,../data/sampled\0\00\0002114d3f.flac,../data/sampled\0\00\0002114d3f.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.200,67.57,131.29
1,../data/sampled\0\00\00079b49f9.flac,../data/sampled\0\00\00079b49f9.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.400,71.20,168.79
2,../data/sampled\0\00\00084ff1da.flac,../data/sampled\0\00\00084ff1da.wav,FLAC,WAV,PCM_16,PCM_16,16000,7.100,91.39,221.92
3,../data/sampled\0\00\000923633f.flac,../data/sampled\0\00\000923633f.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.100,84.85,128.17
4,../data/sampled\0\00\0009d31e8f.flac,../data/sampled\0\00\0009d31e8f.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.300,96.10,165.67
...,...,...,...,...,...,...,...,...,...,...
18384,../data/sampled\9\9f\9fe3d15862.flac,../data/sampled\9\9f\9fe3d15862.wav,FLAC,WAV,PCM_16,PCM_16,16000,8.800,136.22,275.04
18385,../data/sampled\9\9f\9ff3410471.flac,../data/sampled\9\9f\9ff3410471.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.758,107.56,180.00
18386,../data/sampled\9\9f\9ff6f34230.flac,../data/sampled\9\9f\9ff6f34230.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.500,98.99,171.92
18387,../data/sampled\9\9f\9ff789f862.flac,../data/sampled\9\9f\9ff789f862.wav,FLAC,WAV,PCM_16,PCM_16,16000,3.700,62.33,115.67


In [3]:
def sampling(wav_path: str) -> tuple[np.ndarray, int]:
    samples, sr = sf.read(wav_path, dtype="float64")

    if samples.ndim == 2:       # stereo jadi mono 
        samples = samples[:, 0]

    return samples, sr

In [4]:
def quantization(samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Step 1 – level rentang -32768 sampai +32767
    max_level = 32_767
    min_level = -32_768

    # Step 2 – kode PCM: amplitudo kontinu × 32767 kemudian bulatkan
    kode_pcm = np.clip(np.round(samples * max_level), min_level, max_level).astype(int)

    # Step 3 – rekonstruksi kode PCM ÷ 32767 = amplitudo diskrit
    quantized = kode_pcm / max_level

    return kode_pcm, quantized 

In [5]:
all_paths = [
    os.path.abspath(os.path.join(root_dir, file))
    for root_dir, dirs, files in os.walk(sampled_root)
    for file in files
]
print(all_paths)

['d:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\0002114d3f.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\00079b49f9.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\00084ff1da.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\000923633f.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\0009d31e8f.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\000e2b3102.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\000e78988a.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\00175a0bc1.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\00197e04b3.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\001a373938.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\001dcf14f7.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\0020c37d51.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\00210b8a68.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\0023bd2ccf.wav', 'd:\\doc\\2026\\FINAL CODE\\data\\sampled\\0\\00\\002457231d.

In [6]:
hasil_quantization = []
hasil_sampling =[]
for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
    
    samples, sr = sampling(wav_path)

    kode_pcm, quantized = quantization(samples)
    wave = os.path.splitext(os.path.basename(wav_path))[0]
    hasil_sampling.append({
                "Id"   : wave,
                "Sample Rate" : f" {sr} Hz",
                "Jumlah Sampel" : f"{len(samples)}",
                "Durasi"     : f"{len(samples)/sr:.3f} detik",
                "Contoh Sampling" : (samples[:5])
            })

    hasil_quantization.append({
        "Id"      : wave,
        "Sample Rate" : f" {sr} Hz",
        "Jumlah Sampel" : f"{len(samples)}",
        "Durasi"     : f"{len(samples)/sr:.3f} detik",
        "Contoh Sampling"   : samples[:5].tolist(),     # amplitudo kontinu
        "Contoh kode PCM"   : kode_pcm[:5].tolist(),   # hasil step 2
        "Contoh Quantized"  : quantized[:5].tolist(),  # hasil step 3
    })
df_sampling = pd.DataFrame(hasil_sampling)
df_sampling.to_csv("HasilSampling.csv", index=False)
df_quantization = pd.DataFrame(hasil_quantization)
df_quantization.to_csv("HasilQuantization.csv", index=False)

In [7]:
df_sampling

,Id,Sample Rate,Jumlah Sampel,Durasi,Contoh Sampling
0,0002114d3f,16000 Hz,67200,4.200 detik,"[-0.00201416015625, -0.002349853515625, -0.002..."
1,00079b49f9,16000 Hz,86400,5.400 detik,"[-0.00067138671875, -0.000762939453125, -0.001..."
2,00084ff1da,16000 Hz,113600,7.100 detik,"[0.0001220703125, 0.0001220703125, 6.103515625..."
3,000923633f,16000 Hz,65600,4.100 detik,"[0.005279541015625, 0.001953125, 0.00192260742..."
4,0009d31e8f,16000 Hz,84800,5.300 detik,"[0.002105712890625, 0.003204345703125, 0.00408..."
...,...,...,...,...,...
18384,9fe3d15862,16000 Hz,140800,8.800 detik,"[0.0008544921875, 0.00146484375, 0.00164794921..."
18385,9ff3410471,16000 Hz,92136,5.758 detik,"[0.00146484375, 0.0050048828125, -0.0017395019..."
18386,9ff6f34230,16000 Hz,88000,5.500 detik,"[0.0020751953125, 0.0023193359375, 0.001739501..."
18387,9ff789f862,16000 Hz,59200,3.700 detik,"[0.000701904296875, 0.00079345703125, 0.001037..."


In [8]:
df_quantization

,Id,Sample Rate,Jumlah Sampel,Durasi,Contoh Sampling,Contoh kode PCM,Contoh Quantized
0,0002114d3f,16000 Hz,67200,4.200 detik,"[-0.00201416015625, -0.002349853515625, -0.002...","[-66, -77, -69, -35, -10]","[-0.0020142216254158147, -0.002349925229651784..."
1,00079b49f9,16000 Hz,86400,5.400 detik,"[-0.00067138671875, -0.000762939453125, -0.001...","[-22, -25, -35, -34, -31]","[-0.0006714072084719382, -0.000762962736899929..."
2,00084ff1da,16000 Hz,113600,7.100 detik,"[0.0001220703125, 0.0001220703125, 6.103515625...","[4, 4, 2, 13, 10]","[0.00012207403790398877, 0.0001220740379039887..."
3,000923633f,16000 Hz,65600,4.100 detik,"[0.005279541015625, 0.001953125, 0.00192260742...","[173, 64, 63, 136, 195]","[0.005279702139347514, 0.0019531846064638203, ..."
4,0009d31e8f,16000 Hz,84800,5.300 detik,"[0.002105712890625, 0.003204345703125, 0.00408...","[69, 105, 134, 181, 201]","[0.0021057771538438063, 0.003204443494979705, ..."
...,...,...,...,...,...,...,...
18384,9fe3d15862,16000 Hz,140800,8.800 detik,"[0.0008544921875, 0.00146484375, 0.00164794921...","[28, 48, 54, 82, 113]","[0.0008545182653279214, 0.0014648884548478652,..."
18385,9ff3410471,16000 Hz,92136,5.758 detik,"[0.00146484375, 0.0050048828125, -0.0017395019...","[48, 164, -57, -381, -430]","[0.0014648884548478652, 0.0050050355540635395,..."
18386,9ff6f34230,16000 Hz,88000,5.500 detik,"[0.0020751953125, 0.0023193359375, 0.001739501...","[68, 76, 57, 68, 80]","[0.002075258644367809, 0.0023194067201757866, ..."
18387,9ff789f862,16000 Hz,59200,3.700 detik,"[0.000701904296875, 0.00079345703125, 0.001037...","[23, 26, 34, 17, 33]","[0.0007019257179479354, 0.000793481246375927, ..."


In [ ]:
def voice_activity_detection(
    quantized  : np.ndarray,
    sr         : int,
    frame_ms   : int = 300,     # durasi frame: 300 ms menghasilkan 480 sampel (dokumen)
    snr_min    : float = 6.0,   # ambang batas bawah  (dokumen)
    snr_max    : float = 10.0,  # ambang batas atas   (dokumen)
) -> tuple[np.ndarray, list[dict]]:

    #Bagi sinyal menjadi frame 
    frame_size = int(sr * frame_ms / 1000)      # 16000 * 300/1000 = 480
    frames     = [
        quantized[i : i + frame_size]
        for i in range(0, len(quantized), frame_size)
        if len(quantized[i : i + frame_size]) == frame_size  # buang frame tidak lengkap
    ]

    frame_results = []
    voice_frames  = []

    for idx, frame in enumerate(frames):

        # Step 1: Hitung Energi per Frame 
        # Dokumen: "Setiap nilai amplitudo diskrit dipangkatkan dan ditotalkan"
        # ∑x² = 0,0000001341 + ... + 0,0000004107 = 0,0000026934
        energi = np.sum(frame ** 2)

        # Step 2: Perhitungan Daya 
        # Dokumen: P = ∑x² / N  =  0,0000026934 / 20  = 0,0000001347
        N    = len(frame)
        daya = energi / N

        # Step 3: Estimasi Daya 
        # Dokumen: P = (1/32767)² / 12  =  0,0000000009 / 12  = 0,0000000001
        delta         = 1 / 32_767
        estimasi_daya = (delta ** 2) / 12

        # Step 4: Rasio Signal to Noise
        # Dokumen: SNR = daya / estimasi_daya  =  0,0000001347 / 0,0000000001  = 1735,2
        snr = daya / estimasi_daya if estimasi_daya > 0 else 0

        # Step 5: Konversi SNR ke Logaritmik (dB) 
        # Dokumen: SNR = 10 × log10(1735,2)  =  3,2393495390
        snr_db = 10 * np.log10(snr) if snr > 0 else 0

        # Step 6: Perbandingan terhadap Ambang Batas 
        # Dokumen: ambang batas 6 - 10
        # SNR > snr_max -> frame wicara
        # SNR < snr_min -> frame noise
        if snr_db > snr_max:
            label = "wicara"
            voice_frames.append(frame)
        elif snr_db < snr_min:
            label = "noise"
        else:
            label = "tidak pasti"

        frame_results.append({
            "frame_idx"    : idx,
            "energi"       : round(energi, 10),
            "daya"         : round(daya, 10),
            "estimasi_daya": round(estimasi_daya, 10),
            "snr"          : round(snr, 4),
            "snr_db"       : round(snr_db, 10),
            "label"        : label,
        })

    # Gabungkan semua frame wicara menjadi satu sinyal bersih
    sinyal_bersih = np.concatenate(voice_frames) if voice_frames else np.array([])

    return sinyal_bersih, frame_results

In [12]:
hasil_sampling     = []
hasil_quantization = []
hasil_vad          = []

for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
    try:
        audio_id = os.path.splitext(os.path.basename(wav_path))[0]

        # SAMPLING 
        samples, sr = sampling(wav_path)

        hasil_sampling.append({
            "Id"              : audio_id,
            "Sample Rate"     : f"{sr} Hz",
            "Jumlah Sampel"   : len(samples),
            "Durasi"          : f"{len(samples)/sr:.3f} detik",
            "Contoh Sampling" : samples[:5].tolist(),
        })

        # QUANTIZATION 
        kode_pcm, quantized = quantization(samples)

        hasil_quantization.append({
            "Id"              : audio_id,
            "Sample Rate"     : f"{sr} Hz",
            "Jumlah Sampel"   : len(samples),
            "Durasi"          : f"{len(samples)/sr:.3f} detik",
            "Contoh Sampling" : samples[:5].tolist(),
            "Contoh Kode PCM" : kode_pcm[:5].tolist(),
            "Contoh Quantized": quantized[:5].tolist(),
        })

        # VOICE ACTIVITY DETECTION 
        sinyal_bersih, frame_results = voice_activity_detection(quantized, sr)

        df_frame = pd.DataFrame(frame_results)
        hasil_vad.append({
            "Id"             : audio_id,
            "Total Frame"    : len(frame_results),
            "Frame Wicara"   : df_frame[df_frame["label"] == "wicara"].shape[0],
            "Frame Noise"    : df_frame[df_frame["label"] == "noise"].shape[0],
            "Frame Tdk Pasti": df_frame[df_frame["label"] == "tidak pasti"].shape[0],
            "Sinyal Bersih"  : sinyal_bersih[:5].tolist() if len(sinyal_bersih) > 0 else [],
        })
    except Exception as e:
        print(f"Error pada file : {wav_path} : {e}")

df_sampling     = pd.DataFrame(hasil_sampling)
df_quantization = pd.DataFrame(hasil_quantization)
df_vad          = pd.DataFrame(hasil_vad)

df_sampling.to_csv("HasilSampling.csv",     index=False)
df_quantization.to_csv("HasilQuantization.csv", index=False)
df_vad.to_csv("HasilVAD.csv",               index=False)

print(f"\n{'='*40}")
print(f"HasilSampling.csv     -> {len(df_sampling)} file")
print(f"HasilQuantization.csv -> {len(df_quantization)} file")
print(f"HasilVAD.csv          -> {len(df_vad)} file")

Error pada file : ../data/sampled\5\59\5980ab3cf8.wav : 'label'

HasilSampling.csv     -> 18389 file
HasilQuantization.csv -> 18389 file
HasilVAD.csv          -> 18388 file


In [14]:
df_vad

,Id,Total Frame,Frame Wicara,Frame Noise,Frame Tdk Pasti,Sinyal Bersih
0,0002114d3f,14,14,0,0,"[-0.0020142216254158147, -0.002349925229651784..."
1,00079b49f9,18,18,0,0,"[-0.0006714072084719382, -0.000762962736899929..."
2,00084ff1da,23,23,0,0,"[0.00012207403790398877, 0.0001220740379039887..."
3,000923633f,13,13,0,0,"[0.005279702139347514, 0.0019531846064638203, ..."
4,0009d31e8f,17,17,0,0,"[0.0021057771538438063, 0.003204443494979705, ..."
...,...,...,...,...,...,...
18383,9fe3d15862,29,29,0,0,"[0.0008545182653279214, 0.0014648884548478652,..."
18384,9ff3410471,19,19,0,0,"[0.0014648884548478652, 0.0050050355540635395,..."
18385,9ff6f34230,18,18,0,0,"[0.002075258644367809, 0.0023194067201757866, ..."
18386,9ff789f862,12,12,0,0,"[0.0007019257179479354, 0.000793481246375927, ..."
